# Day 2 — Connecting to Data Sources

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-02-connecting-data.ipynb)

**Course:** Great Expectations for Data Quality  
**Day:** 2 of 5  
**Badge:** Learn

---

## What you will learn

By the end of this notebook you will be able to:

- Connect a Pandas datasource to a CSV file and inspect its batches
- Connect a SQLite datasource using an in-memory engine
- Understand the relationship between DataAsset, BatchRequest, and Batch
- Use `validator.head()` to confirm a batch loaded correctly
- Explain key `BatchRequest` parameters: `data_asset_name`, `datasource_name`, `batch_slice`

## 0 — Setup

Install the required packages. We need `great-expectations` for validation and `sqlalchemy` for the SQLite datasource connector.

In [ ]:
!pip install great-expectations sqlalchemy -q

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np
import tempfile
import os

print('great_expectations version:', gx.__version__)

# Create a fresh EphemeralDataContext for this notebook
context = gx.get_context()
print('Context type:', type(context).__name__)

## 1 — Datasource and Data Asset Concepts

Before connecting to data it is important to understand the two levels of abstraction GX introduces:

### Datasource
A **Datasource** encapsulates the connection details for a backend — file system, Pandas, Spark, or a SQL database. It knows the *how* of connecting but nothing about *which* data to read.

### DataAsset
A **DataAsset** is a named, logical pointer to a specific dataset within a Datasource — a particular CSV file, a database table, or a named query. You define the asset once and generate BatchRequests many times (one per pipeline run or date partition).

### Batch
A **Batch** is a concrete, in-memory slice of data materialised from a BatchRequest at runtime. It is the thing the Validator actually inspects.

```
Datasource  (knows HOW to connect)
  └── DataAsset  (named WHAT to read)
        └── BatchRequest  (parameters for ONE slice)
              └── Batch  (materialised data in memory)
                    └── Validator  (runs Expectations against the Batch)
```

> **Reference:** [Datasource and Data Asset concepts](https://docs.greatexpectations.io/docs/reference/learn/conceptual_guides/datasource)

## 2 — Pandas Datasource with a CSV File

The most common starting point for GX is a Pandas datasource reading local files. We will:

1. Generate a synthetic DataFrame and write it to a temporary CSV
2. Register a `PandasFilesystemDatasource` pointing to the directory
3. Add a `CSVAsset` for the specific file
4. Build a `BatchRequest` and load a `Validator`

> **Reference:** [Connect to file system data](https://docs.greatexpectations.io/docs/core/connect_to_data/file_system/)

In [ ]:
# Generate a synthetic sales DataFrame
np.random.seed(0)
n = 200

sales_df = pd.DataFrame({
    'sale_id':    range(1, n + 1),
    'product':    np.random.choice(['Widget', 'Gadget', 'Doohickey'], n),
    'quantity':   np.random.randint(1, 50, n),
    'unit_price': np.round(np.random.uniform(5.0, 200.0, n), 2),
    'customer_email': [f'cust{i}@shop.example' for i in range(1, n + 1)],
    'region':     np.random.choice(['North', 'South', 'East', 'West'], n),
})

print('sales_df shape:', sales_df.shape)
print(sales_df.head(5))

In [ ]:
# Write the DataFrame to a temporary CSV file
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, 'sales.csv')
sales_df.to_csv(csv_path, index=False)

print('CSV written to:', csv_path)
print('File size (bytes):', os.path.getsize(csv_path))

In [ ]:
# Register a PandasFilesystemDatasource
pandas_ds = context.data_sources.add_pandas_filesystem(
    name='sales_filesystem',
    base_directory=tmp_dir,
)

print('Datasource name:', pandas_ds.name)
print('Datasource type:', type(pandas_ds).__name__)

In [ ]:
import re

# Add a CSVAsset pointing to the sales.csv file
csv_asset = pandas_ds.add_csv_asset(
    name='sales_csv',
    batching_regex=re.compile(r'sales\.csv'),
)

print('Asset name:', csv_asset.name)
print('Asset type:', type(csv_asset).__name__)

In [ ]:
# Build a BatchRequest
batch_request = csv_asset.build_batch_request()

print('BatchRequest datasource_name:', batch_request.datasource_name)
print('BatchRequest data_asset_name:', batch_request.data_asset_name)

# Create an ExpectationSuite
sales_suite = context.suites.add(gx.ExpectationSuite(name='sales_suite'))

# Load a Validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=sales_suite,
)

print()
print('Validator loaded. First 3 rows:')
print(validator.head(3))

## 3 — Understanding BatchRequest Parameters

A `BatchRequest` is a lightweight specification that tells GX *which slice* of a DataAsset to load. Key parameters:

| Parameter | Purpose |
|---|---|
| `datasource_name` | Auto-populated from the asset; identifies the backend |
| `data_asset_name` | Auto-populated from the asset; the logical dataset name |
| `options` | Dict of batch identifiers (e.g. date partitions) for partitioned assets |
| `batch_slice` | A Python slice to select a subset of matched batches by index |

### Batch slice example

When a DataAsset matches multiple files (e.g. `sales_2024_01.csv`, `sales_2024_02.csv`, …), `batch_slice` selects which of those batches to load:

```python
# Load only the most recent batch
batch_request = asset.build_batch_request(batch_slice=-1)

# Load the first three batches
batch_request = asset.build_batch_request(batch_slice=slice(0, 3))
```

In our single-file example there is only one batch, so `batch_slice` has no visible effect — but it becomes essential in production partitioned pipelines.

In [ ]:
# Inspect available batches for the asset
batches = csv_asset.get_batch_list_from_batch_request(batch_request)
print('Number of batches available:', len(batches))

for i, batch in enumerate(batches):
    print(f'  Batch {i}: id={batch.id}')

## 4 — SQLite Datasource

GX supports SQL backends through SQLAlchemy. An SQLite in-memory database is the easiest way to practise SQL datasources without setting up a server.

> **Reference:** [Connect to SQL data](https://docs.greatexpectations.io/docs/core/connect_to_data/sql/)

In [ ]:
from sqlalchemy import create_engine, text

# Create a SQLite in-memory engine and load the sales data into it
engine = create_engine('sqlite://')

with engine.connect() as conn:
    sales_df.to_sql('sales', conn, index=False, if_exists='replace')
    result = conn.execute(text('SELECT COUNT(*) FROM sales'))
    row_count = result.fetchone()[0]

print('Rows loaded into SQLite:', row_count)

In [ ]:
# Register a SQLite datasource using the in-memory connection string
# Note: 'sqlite://' is ephemeral — use 'sqlite:///path/to/file.db' for a persistent DB
sqlite_ds = context.data_sources.add_sqlite(
    name='sales_sqlite',
    connection_string='sqlite://',
)

print('SQLite datasource type:', type(sqlite_ds).__name__)
print('SQLite datasource name:', sqlite_ds.name)

In [ ]:
# Add a TableAsset pointing to the 'sales' table
sql_asset = sqlite_ds.add_table_asset(
    name='sales_table',
    table_name='sales',
)

print('SQL asset name:', sql_asset.name)
print('SQL asset type:', type(sql_asset).__name__)

### Pandas vs SQL Datasources — Side-by-Side Comparison

| Aspect | Pandas Filesystem | SQLite (SQL) |
|---|---|---|
| Asset type | `CSVAsset`, `ParquetAsset`, … | `TableAsset`, `QueryAsset` |
| Connection | Base directory path | SQLAlchemy connection string |
| Batch slicing | File pattern matching | Row-level sampling or query slicing |
| Scale | Fits in memory | Can push down filters to the DB |
| Best for | Local files, S3, GCS | Postgres, MySQL, Snowflake, BigQuery |

The Validator API is **identical** regardless of backend — expectations you write for CSV work unchanged on SQL tables.

## 5 — Quick Validator Inspection Methods

Once you have a `Validator`, several methods help you explore the batch before writing expectations:

In [ ]:
# validator.head() — first N rows (default 5)
print('=== validator.head(3) ===')
print(validator.head(3))

print()
# validator.get_column_value_counts() is available for categorical columns
# Here we just list the column names from the active batch
print('Column names:', list(validator.active_batch.data.columns))

In [ ]:
# Validate that the batch loaded correctly by running a simple row-count expectation
result = validator.expect_table_row_count_to_equal(value=200)
print('Row count expectation success:', result.success)
print('Observed value:', result.result.get('observed_value'))

## Challenge — Add a QueryAsset

GX SQL datasources support `QueryAsset` — an asset defined by an arbitrary SQL query rather than a table name. This is useful when you want to validate a complex join or aggregation.

Complete the tasks below:

1. Use `sqlite_ds.add_query_asset()` to create an asset named `'high_value_sales'` that selects only rows where `unit_price > 100`
2. Build a `BatchRequest` from this asset
3. Create a new `ExpectationSuite` named `'high_value_suite'`
4. Load a `Validator` and call `validator.head()` to confirm the filter worked
5. Print the number of rows in the batch using `expect_table_row_count_to_be_between`

In [ ]:
# Your solution here
# Step 1: add a QueryAsset

# Step 2: build BatchRequest

# Step 3: create ExpectationSuite

# Step 4: load Validator and call head()

# Step 5: check row count


## Day 2 Recap

| Concept | Key takeaway |
|---|---|
| DataAsset | Named logical pointer to data; defined once, queried many times |
| BatchRequest | Lightweight spec for one slice; auto-populated with `build_batch_request()` |
| Batch | Materialised data at runtime; what the Validator actually inspects |
| `batch_slice` | Selects which of multiple matched files/partitions to load |
| Pandas Filesystem | `add_pandas_filesystem()` + `add_csv_asset()` — file-based validation |
| SQLite / SQL | `add_sqlite()` + `add_table_asset()` or `add_query_asset()` — DB validation |
| Validator portability | Same expectation API works across all backends |

---

**Up next — Day 3:** Core Expectations — running built-in checks, chaining expectations, comparing result formats, and writing a utility audit function.